# Explainable and Mobile-Optimised Glaucoma Screening (Kaggle Notebooks edition)

Implementation notebook matching Chapter 3 (Research Methodology) of the project, adapted to run on **Kaggle Notebooks** instead of Colab - no API token needed for datasets already on Kaggle, just 'Add Data' in the UI.

**Pipeline:** data acquisition -> disc/cup-aware preprocessing -> attention-augmented lightweight CNN (EfficientNetB0 + CBAM) -> training -> evaluation -> Grad-CAM -> TensorFlow Lite export.

Run top to bottom. Cells that need you to do something in the Kaggle UI (Add Data, upload a private dataset) are marked **ACTION**.

## 0. Runtime check
Make sure a GPU is attached: right-hand panel -> **Settings** -> **Accelerator** -> `GPU T4 x2` (or P100). Also switch **Internet** to **On** in the same panel - needed for `pip install` and the RIM-ONE DL `git clone` below. Kaggle gives ~30 GPU-hours/week on the free tier, so avoid leaving a GPU session idle.

In [ ]:
import tensorflow as tf
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
assert len(tf.config.list_physical_devices('GPU')) > 0, "No GPU found - check Settings > Accelerator in the right panel"

## 1. Install dependencies
Most of these ship pre-installed on Kaggle's TensorFlow image; this just fills any gaps.

In [ ]:
!pip install -q tf-keras-vis

## 2. Paths
Kaggle Notebooks have two relevant locations:
- `/kaggle/input/...` - **read-only**, where datasets you attach via 'Add Data' get mounted, one folder per dataset.
- `/kaggle/working/...` - **writable**, persists when you 'Save Version' (this is where checkpoints, logs and any data you need to download yourself go).

In [ ]:
import os

INPUT_ROOT = '/kaggle/input'
WORK_DIR = '/kaggle/working'
DATA_DIR = f'{WORK_DIR}/data'       # for anything we download ourselves (RIM-ONE DL via git clone)
CKPT_DIR = f'{WORK_DIR}/checkpoints'
LOG_DIR = f'{WORK_DIR}/logs'
for d in [DATA_DIR, CKPT_DIR, LOG_DIR]:
    os.makedirs(d, exist_ok=True)
print('Attached datasets currently visible under /kaggle/input:')
print(os.listdir(INPUT_ROOT) if os.path.exists(INPUT_ROOT) else '(none attached yet)')

## 3. Configuration
Matches Table 3.3 (proposed hyperparameters).

In [ ]:
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

CONFIG = {
    'input_size': (224, 224),
    'batch_size': 32,
    'warmup_epochs': 15,
    'finetune_epochs': 35,
    'warmup_lr': 1e-4,
    'finetune_lr': 1e-5,
    'dropout': 0.3,
    'backbone': 'EfficientNetB0',   # or 'MobileNetV3Small'
    'early_stopping_patience': 5,
    'seed': SEED,
    'crop_margin': 0.15,  # extra margin kept around the disc/cup bounding box
}
CONFIG

## 4. Dataset acquisition

Four datasets form the training corpus; REFUGE2 is held out entirely for external validation (Section 3.4.4 / 3.11).

| Dataset | How to get it on Kaggle | Access |
|---|---|---|
| G1020 | **ACTION:** click **Add Data** (top right) -> search `glaucoma-datasets` -> add the one by `arnavjain1`. Mounts at `/kaggle/input/glaucoma-datasets/`. | Free Kaggle account, no token needed |
| RIM-ONE DL | Cloned automatically below from GitHub (`miag-ull/rim-one-dl`) | Open |
| ACRIMA | Download from https://figshare.com/s/c2d31f850af14c5b5232, then **ACTION:** upload it as your own **private Kaggle dataset** (kaggle.com -> Create -> New Dataset), then **Add Data** -> search your dataset name and attach it here | Open source, private re-upload on Kaggle |
| REFUGE | Download from https://refuge.grand-challenge.org/REFUGE2Download/ (after registration), then upload as a private Kaggle dataset the same way as ACRIMA | Registration required |
| REFUGE2 | Same page as REFUGE, same private-dataset-upload approach | Registration required |

**Why private re-upload for ACRIMA/REFUGE/REFUGE2?** Kaggle Notebooks can only directly attach data that exists as a Kaggle Dataset - there's no general-purpose internet download helper inside `/kaggle/input`. Uploading once as a private dataset (visible only to you) is the standard way to get outside data into a Kaggle Notebook, and then you can reuse it in every future notebook with one click.

In [ ]:
# Fill these in with the actual folder names after you've attached each dataset via 'Add Data'
# (check the printed list from the previous cell, or re-run it after attaching).
DATASET_PATHS = {
    'g1020': f'{INPUT_ROOT}/glaucoma-datasets',        # from arnavjain1/glaucoma-datasets
    'acrima': f'{INPUT_ROOT}/REPLACE_WITH_YOUR_ACRIMA_DATASET_SLUG',
    'refuge': f'{INPUT_ROOT}/REPLACE_WITH_YOUR_REFUGE_DATASET_SLUG',
    'refuge2': f'{INPUT_ROOT}/REPLACE_WITH_YOUR_REFUGE2_DATASET_SLUG',
    'rim_one_dl': f'{DATA_DIR}/rim_one_dl',             # cloned below, lives in /kaggle/working not /kaggle/input
}
DATASET_PATHS

### 4.1 Clone RIM-ONE DL (GitHub, open access)

In [ ]:
rim_one_dir = DATASET_PATHS['rim_one_dl']
if not os.path.exists(rim_one_dir):
    !git clone --depth 1 https://github.com/miag-ull/rim-one-dl.git {rim_one_dir}
else:
    print('RIM-ONE DL already present at', rim_one_dir)

### 4.2 Verify the attached datasets
Run this after you've used **Add Data** for G1020, ACRIMA, REFUGE and REFUGE2. If a path shows `NOT FOUND`, double-check the exact folder name Kaggle gave it (it's usually the dataset's URL slug) and fix the corresponding entry in `DATASET_PATHS` above.

In [ ]:
for name, path in DATASET_PATHS.items():
    status = 'OK' if os.path.exists(path) else 'NOT FOUND'
    print(f'{name:12s} -> {path}  [{status}]')

### 4.3 A note on folder layout
Kaggle extracts an uploaded dataset zip automatically, so whatever you see under each `DATASET_PATHS` entry above is the dataset's real internal structure - no manual unzip step needed here (unlike the Colab/Drive version of this notebook).

In [ ]:
# Nothing to run in this cell for the Kaggle version - kept as a placeholder so section numbering matches the Colab notebook.
pass

## 5. Build a unified manifest
Each dataset has a different folder/label layout. This section normalises them into one CSV: `image_path, label, mask_path, source, split_hint`.

**You will need to adjust the folder-walking logic below once you see each dataset's actual extracted structure** — run the inspection cell first and fill in the per-dataset parser.

In [ ]:
# Inspect what actually got attached/downloaded before writing parsers
for name, d in DATASET_PATHS.items():
    if os.path.exists(d):
        print(f'--- {name} ({d}) ---')
        for root, dirs, fs in os.walk(d):
            depth = root[len(d):].count(os.sep)
            if depth <= 2:
                print('  ' * depth + os.path.basename(root) + f'/  ({len(fs)} files)')
    else:
        print(f'--- {name}: not available yet at {d} ---')

In [ ]:
import pandas as pd
import glob

manifest_rows = []

def add_rows(image_paths, labels, source, mask_paths=None):
    mask_paths = mask_paths or [None] * len(image_paths)
    for img, lbl, msk in zip(image_paths, labels, mask_paths):
       manifest_rows.append({'image_path': img, 'label': int(lbl), 'mask_path': msk, 'source': source})

# --- RIM-ONE DL: images under Images/Glaucoma and Images/Normal (verify against the inspection cell above) ---
rim_dir = DATASET_PATHS['rim_one_dl']
rim_glaucoma = (glob.glob(f'{rim_dir}/**/Glaucoma*/*.png', recursive=True) + 
               glob.glob(f'{rim_dir}/**/Glaucoma*/*.jpg', recursive=True))
rim_normal = (glob.glob(f'{rim_dir}/**/Normal*/*.png', recursive=True) + 
             glob.glob(f'{rim_dir}/**/Normal*/*.jpg', recursive=True))
if rim_glaucoma or rim_normal:
    add_rows(rim_glaucoma, [1] * len(rim_glaucoma), 'rim_one_dl')
    add_rows(rim_normal, [0] * len(rim_normal), 'rim_one_dl')
    print(f'RIM-ONE DL: {len(rim_glaucoma)} glaucoma, {len(rim_normal)} normal')
else:
    print('RIM-ONE DL: no images matched - check folder names from the inspection cell and adjust the glob patterns')

# --- G1020: typically has an Images/ folder plus a label CSV - adjust once you see the real layout ---
g1020_dir = DATASET_PATHS['g1020']
g1020_csv_candidates = glob.glob(f'{g1020_dir}/**/*.csv', recursive=True)
print('G1020 CSV candidates found:', g1020_csv_candidates)
# Example once you know the real CSV columns:
# g1020_df = pd.read_csv(g1020_csv_candidates[0])
# add_rows(g1020_df['image_path'].tolist(), g1020_df['binaryLabels'].tolist(), 'g1020')

# --- ACRIMA: filenames usually encode the label, e.g. '..._g_...' = glaucoma, '..._n_...' = normal ---
acrima_dir = DATASET_PATHS['acrima']
acrima_images = (glob.glob(f'{acrima_dir}/**/*.jpg', recursive=True) + 
                 glob.glob(f'{acrima_dir}/**/*.png', recursive=True))
if acrima_images:
    acrima_labels = [1 if '_g_' in os.path.basename(p).lower() or 'glaucoma' in p.lower() else 0 for p in acrima_images]
    add_rows(acrima_images, acrima_labels, 'acrima')
    print(f'ACRIMA: {sum(acrima_labels)} glaucoma, {len(acrima_labels) - sum(acrima_labels)} normal')
else:
    print('ACRIMA: not attached yet, or glob pattern needs adjusting')

print(f'\nTotal rows so far: {len(manifest_rows)}')

**Note:** the G1020 and REFUGE/REFUGE2 parsers above are left as templates because their exact folder/CSV layout can only be confirmed after download (dataset hosts occasionally reorganise their archives). Run the inspection cell, look at the printed tree, then fill in the three commented-out blocks before continuing. REFUGE/REFUGE2 ship an official CSV of labels (`glaucoma`/`non-glaucoma`) plus disc/cup segmentation masks - wire those into `add_rows(..., mask_paths=...)` the same way.

In [ ]:
manifest = pd.DataFrame(manifest_rows)
manifest = manifest.drop_duplicates(subset=['image_path']).reset_index(drop=True)
manifest.to_csv(f'{WORK_DIR}/manifest.csv', index=False)
print(manifest['source'].value_counts())
print(manifest['label'].value_counts())
manifest.head()

## 6. Disc/cup-aware preprocessing (Section 3.5)

- If a segmentation mask is available, crop to the disc/cup bounding box with a margin (keeps peripapillary context, per Hemelings et al., 2021).
- If no mask is available, fall back to a brightness-based optic-disc localisation (the disc is the brightest compact region in the green channel) and crop around that.
- All images are resized to 224x224 and normalised to [0, 1].

In [ ]:
import cv2

def crop_from_mask(image, mask, margin_frac=CONFIG['crop_margin']):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return None
    x1, x2, y1, y2 = xs.min(), xs.max(), ys.min(), ys.max()
    w, h = x2 - x1, y2 - y1
    mx, my = int(w * margin_frac), int(h * margin_frac)
    x1, y1 = max(0, x1 - mx), max(0, y1 - my)
    x2, y2 = min(image.shape[1], x2 + mx), min(image.shape[0], y2 + my)
    return image[y1:y2, x1:x2]

def crop_disc_fallback(image, margin_frac=CONFIG['crop_margin']):
    """Brightness-based optic disc localisation when no mask is provided."""
    green = image[:, :, 1]
    blurred = cv2.GaussianBlur(green, (25, 25), 0)
    _, thresh = cv2.threshold(blurred, np.percentile(blurred, 97), 255, cv2.THRESH_BINARY)
    thresh = thresh.astype(np.uint8)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return image  # fall back to the full image
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    mx, my = int(w * margin_frac * 3), int(h * margin_frac * 3)  # wider margin since this is a coarse estimate
    x1, y1 = max(0, x - mx), max(0, y - my)
    x2, y2 = min(image.shape[1], x + w + mx), min(image.shape[0], y + h + my)
    crop = image[y1:y2, x1:x2]
    return crop if crop.size > 0 else image

def preprocess_image(image_path, mask_path=None, size=CONFIG['input_size']):
    image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    cropped = None
    if mask_path and os.path.exists(mask_path):
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if mask is not None and mask.shape[:2] == image.shape[:2]:
            cropped = crop_from_mask(image, mask)
    if cropped is None:
        cropped = crop_disc_fallback(image)
    resized = cv2.resize(cropped, size, interpolation=cv2.INTER_AREA)
    return resized.astype(np.float32) / 255.0

# Quick visual sanity check on a handful of images
import matplotlib.pyplot as plt
sample = manifest.sample(min(6, len(manifest)), random_state=SEED) if len(manifest) else None
if sample is not None and len(sample):
    fig, axes = plt.subplots(1, len(sample), figsize=(18, 3))
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = preprocess_image(row['image_path'], row.get('mask_path'))
        ax.imshow(img)
        ax.set_title(f"{row['source']}\nlabel={row['label']}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('Manifest is empty - finish Section 5 first.')

## 7. Stratified split and tf.data pipeline (Section 3.4.4, 3.5.7)
70% train / 15% validation / 15% internal test, stratified by source dataset and class.

In [ ]:
from sklearn.model_selection import train_test_split

manifest['strata'] = manifest['source'] + '_' + manifest['label'].astype(str)
train_df, temp_df = train_test_split(manifest, test_size=0.30, stratify=manifest['strata'], random_state=SEED)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df['strata'], random_state=SEED)
print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')

train_df.to_csv(f'{WORK_DIR}/train.csv', index=False)
val_df.to_csv(f'{WORK_DIR}/val.csv', index=False)
test_df.to_csv(f'{WORK_DIR}/test.csv', index=False)

In [ ]:
def augment(image):
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_brightness(image, max_delta=0.15)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    image = tf.image.random_crop(
        tf.image.resize_with_pad(image, 256, 256), size=[*CONFIG['input_size'], 3]
    )
    return tf.clip_by_value(image, 0.0, 1.0)

def make_dataset(df, training=False, batch_size=CONFIG['batch_size']):
    paths = df['image_path'].tolist()
    masks = [m if isinstance(m, str) else '' for m in df['mask_path'].tolist()]
    labels = df['label'].astype(np.float32).tolist()

    def _load(path, mask_path, label):
        path = path.numpy().decode('utf-8')
        mask_path = mask_path.numpy().decode('utf-8') or None
        img = preprocess_image(path, mask_path)
        return img, label

    def _tf_load(path, mask_path, label):
        img, label = tf.py_function(_load, [path, mask_path, label], [tf.float32, tf.float32])
        img.set_shape([*CONFIG['input_size'], 3])
        label.set_shape([])
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((paths, masks, labels))
    if training:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)
    ds = ds.map(_tf_load, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        ds = ds.map(lambda x, y: (augment(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

if len(train_df):
    train_ds = make_dataset(train_df, training=True)
    val_ds = make_dataset(val_df, training=False)
    test_ds = make_dataset(test_df, training=False)
    print('Pipelines built.')
else:
    print('Build the manifest first (Section 5) before creating datasets.')

## 8. Model architecture: CBAM-augmented lightweight backbone (Section 3.6)
Input -> pretrained backbone (frozen initially) -> CBAM (channel + spatial attention) -> GAP -> Dropout -> Dense(1, sigmoid).

In [ ]:
from tensorflow.keras import layers, models, applications

def cbam_block(input_feature, ratio=8, name='cbam'):
    """Convolutional Block Attention Module (Woo et al., 2018)."""
    channels = input_feature.shape[-1]

    # --- Channel attention ---
    avg_pool = layers.GlobalAveragePooling2D(keepdims=True)(input_feature)
    max_pool = layers.GlobalMaxPooling2D(keepdims=True)(input_feature)

    shared_dense_1 = layers.Dense(channels // ratio, activation='relu', name=f'{name}_ch_dense1')
    shared_dense_2 = layers.Dense(channels, name=f'{name}_ch_dense2')

    avg_out = shared_dense_2(shared_dense_1(avg_pool))
    max_out = shared_dense_2(shared_dense_1(max_pool))
    channel_attention = layers.Activation('sigmoid', name=f'{name}_ch_sigmoid')(avg_out + max_out)
    x = layers.Multiply(name=f'{name}_ch_scale')([input_feature, channel_attention])

    # --- Spatial attention ---
    avg_spatial = tf.reduce_mean(x, axis=-1, keepdims=True)
    max_spatial = tf.reduce_max(x, axis=-1, keepdims=True)
    concat = layers.Concatenate(axis=-1)([avg_spatial, max_spatial])
    spatial_attention = layers.Conv2D(1, kernel_size=7, padding='same', activation='sigmoid',
                                       name=f'{name}_sp_conv')(concat)
    out = layers.Multiply(name=f'{name}_sp_scale')([x, spatial_attention])
    return out

def build_model(backbone_name=CONFIG['backbone'], input_shape=(*CONFIG['input_size'], 3), dropout=CONFIG['dropout']):
    inputs = layers.Input(shape=input_shape, name='input_image')

    if backbone_name == 'EfficientNetB0':
        preprocess = applications.efficientnet.preprocess_input
        base = applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=input_shape)
    elif backbone_name == 'MobileNetV3Small':
        preprocess = applications.mobilenet_v3.preprocess_input
        base = applications.MobileNetV3Small(include_top=False, weights='imagenet', input_shape=input_shape)
    else:
        raise ValueError(f'Unknown backbone: {backbone_name}')

    base.trainable = False  # frozen for the warm-up phase (Section 3.6.6)

    x = layers.Rescaling(255.0)(inputs)  # undo the 0-1 scaling from preprocess_image, then re-apply backbone's own preprocessing
    x = preprocess(x)
    x = base(x, training=False)
    x = cbam_block(x, name='cbam')
    x = layers.GlobalAveragePooling2D(name='gap')(x)
    x = layers.Dropout(dropout, name='dropout')(x)
    outputs = layers.Dense(1, activation='sigmoid', name='glaucoma_probability')(x)

    model = models.Model(inputs, outputs, name=f'glaucoma_{backbone_name}_cbam')
    return model, base

model, backbone = build_model()
model.summary()

## 9. Class weights and loss (Section 3.5.6)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

if len(train_df):
    classes = np.array([0, 1])
    weights = compute_class_weight('balanced', classes=classes, y=train_df['label'].values)
    class_weight = {0: weights[0], 1: weights[1]}
    print('Class weights:', class_weight)
else:
    class_weight = {0: 1.0, 1: 1.0}

## 10. Two-phase training (Section 3.6.6)
Phase 1: frozen backbone, train the attention block and head.
Phase 2: unfreeze the top backbone layers and fine-tune at a lower learning rate.

In [ ]:
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc'),
    tf.keras.metrics.AUC(name='pr_auc', curve='PR'),
]

def get_callbacks(tag):
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=CONFIG['early_stopping_patience'],
                                          restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7),
        tf.keras.callbacks.ModelCheckpoint(f'{CKPT_DIR}/best_{tag}.keras', monitor='val_loss',
                                            save_best_only=True),
        tf.keras.callbacks.CSVLogger(f'{LOG_DIR}/history_{tag}.csv'),
    ]

if len(train_df):
    # Phase 1: warm-up
    model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG['warmup_lr']),
                   loss='binary_crossentropy', metrics=METRICS)
    history_warmup = model.fit(train_ds, validation_data=val_ds, epochs=CONFIG['warmup_epochs'],
                                class_weight=class_weight, callbacks=get_callbacks('warmup'))

    # Phase 2: fine-tune (unfreeze top of the backbone)
    backbone.trainable = True
    for layer in backbone.layers[:-30]:
        layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(CONFIG['finetune_lr']),
                   loss='binary_crossentropy', metrics=METRICS)
    history_finetune = model.fit(train_ds, validation_data=val_ds, epochs=CONFIG['finetune_epochs'],
                                  class_weight=class_weight, callbacks=get_callbacks('finetune'))
else:
    print('No training data yet - finish Sections 4-7 first.')

## 11. Evaluation (Section 3.9)
Full metric suite: accuracy, precision, recall, specificity, F1, balanced accuracy, MCC, Cohen's Kappa, ROC-AUC, PR-AUC, log loss.

In [ ]:
from sklearn.metrics import (confusion_matrix, matthews_corrcoef, cohen_kappa_score,
                              roc_auc_score, average_precision_score, log_loss,
                              f1_score, balanced_accuracy_score, classification_report)

def evaluate(model, dataset, df, tag='test'):
    y_true = df['label'].values
    y_prob = model.predict(dataset).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    results = {
        'accuracy': (tp + tn) / len(y_true),
        'precision': tp / (tp + fp) if (tp + fp) else 0.0,
        'recall_sensitivity': tp / (tp + fn) if (tp + fn) else 0.0,
        'specificity': specificity,
        'f1': f1_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'mcc': matthews_corrcoef(y_true, y_pred),
        'cohens_kappa': cohen_kappa_score(y_true, y_pred),
        'roc_auc': roc_auc_score(y_true, y_prob),
        'pr_auc': average_precision_score(y_true, y_prob),
        'log_loss': log_loss(y_true, y_prob),
    }
    print(f'--- {tag} results ---')
    for k, v in results.items():
        print(f'{k:>20s}: {v:.4f}')
    print('\n', classification_report(y_true, y_pred, target_names=['Non-glaucoma', 'Glaucoma']))
    return results, y_prob, y_pred

if len(test_df):
    test_results, test_probs, test_preds = evaluate(model, test_ds, test_df, tag='internal test')

## 12. External validation on REFUGE2 (Section 3.11)
REFUGE2 is never used for training - only this final evaluation.

In [ ]:
refuge2_dir = DATASET_PATHS['refuge2']
refuge2_images = (glob.glob(f'{refuge2_dir}/**/*.jpg', recursive=True) + 
                  glob.glob(f'{refuge2_dir}/**/*.png', recursive=True))
print(f'Found {len(refuge2_images)} REFUGE2 images (build the labelled manifest for this set the same way as Section 5 once the official labels are confirmed).')
# refuge2_df = ...  # build the same way as the training manifest once its CSV layout is confirmed
# refuge2_ds = make_dataset(refuge2_df, training=False)
# external_results, _, _ = evaluate(model, refuge2_ds, refuge2_df, tag='REFUGE2 external validation')

## 13. Grad-CAM explainability (Section 2.1.6 / 3.8)

In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_layer_name='cbam_sp_scale'):
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]
    grads = tape.gradient(loss, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()

def show_gradcam(image_path, mask_path=None, model=model, alpha=0.4):
    img = preprocess_image(image_path, mask_path)
    img_array = np.expand_dims(img, axis=0)
    heatmap = make_gradcam_heatmap(img_array, model)
    heatmap_resized = cv2.resize(heatmap, CONFIG['input_size'])
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_color = cv2.cvtColor(heatmap_color, cv2.COLOR_BGR2RGB) / 255.0
    overlay = img * (1 - alpha) + heatmap_color * alpha

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img); axes[0].set_title('Input'); axes[0].axis('off')
    axes[1].imshow(heatmap_resized, cmap='jet'); axes[1].set_title('Grad-CAM'); axes[1].axis('off')
    axes[2].imshow(overlay); axes[2].set_title('Overlay'); axes[2].axis('off')
    plt.tight_layout(); plt.show()
    prob = model.predict(img_array, verbose=0)[0][0]
    print(f'Predicted glaucoma probability: {prob:.3f}')

if len(test_df):
    row = test_df.iloc[0]
    show_gradcam(row['image_path'], row.get('mask_path'))

## 14. TensorFlow Lite conversion and deployment metrics (Section 3.8.2 / 3.9.4)

In [ ]:
import time

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = f'{CKPT_DIR}/glaucoma_model.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

model_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print(f'TFLite model size: {model_size_mb:.2f} MB')
print(f'Total parameters: {model.count_params():,}')

# Rough on-device inference time estimate (CPU, this Colab instance - not a phone, but indicative)
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

dummy = np.random.rand(1, *CONFIG['input_size'], 3).astype(np.float32)
times = []
for _ in range(20):
    start = time.time()
    interpreter.set_tensor(input_details[0]['index'], dummy)
    interpreter.invoke()
    _ = interpreter.get_tensor(output_details[0]['index'])
    times.append(time.time() - start)
print(f'Mean inference time: {np.mean(times) * 1000:.1f} ms (over {len(times)} runs, Colab CPU - re-benchmark on an actual phone before citing this in Chapter 4)')

## 15. Save everything
Keras model, TFLite model, and metric results all land in Google Drive so they survive the Colab session ending.

In [ ]:
import json

model.save(f'{CKPT_DIR}/glaucoma_model.keras')

if len(test_df):
    with open(f'{LOG_DIR}/test_results.json', 'w') as f:
        json.dump({k: float(v) for k, v in test_results.items()}, f, indent=2)

print('Saved to', CKPT_DIR)
print('Done.')